# Actividad 2
__Curso:__ Tópicos avanzados en Inteligencia Artificial 1

__Programa:__ MIA 2-2025

__Profesor:__ Anthony D. Cho

__Ayudante corrector:__ Luis Oliveros

## Instrucciones
* La actividad debe ser realizada por los grupos capstones
* Por favor responder en este mismo notebook (una entrega por grupo)
* Renombrar el archivo agregando el apellido de las y los integrantes, por ejemplo actividad2_Sakuragi_Mitsui_Rukagua_Sendoh.ipynb
* Subir el archivo al link de entrega Actividad 2 en webcursos que será habilitado

__Fecha de entrega:__ Fecha límite de entrega 06 de septiembre de 2026 - 23:59 horas chile.

__Integrantes:__ (RUT, Nombre y Apellido)

* 13.257.556-8, Ricardo Lopez
* 16.789.149-7, Camilo Muñoz


In [1]:
## Descomente la siguiente linea si es necesrio instalar la siguiente libreria
#python -m pip install kagglehub

## Librerias

In [ ]:
import kagglehub
from pathlib import Path
import matplotlib.pyplot as plt
from pandas import read_csv

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

## Agregar las otras librerias que necesiten


## Funciones personalizadas

In [ ]:
def plot_history(history, width=12, height=6):
  """
  DESCRIPTION:
    History performance of the keras model

  INPUT:
    @param history: history of performance of fitted model
    @type history: tensorflow.python.keras.callbacks.History

  OUTPUT:
    A graphic
  """

  ## Metrics keys stored in tensorflow object
  keys = list(history.history.keys())

  ## Number of epoch used for fit the model
  epoch = range(1, len(history.epoch) +1)

  ## Check if validation set was used.
  withValidation = False
  for key in keys:
    if 'val' in key:
      withValidation = True

  ## Number of metrics
  nMetrics = len(keys)
  if withValidation:
    nMetrics = nMetrics//2

  ## Plot-space instance
  plt.figure(figsize=(width, height))

  for i in range(nMetrics):
    plt.subplot(nMetrics, 1, i+1)

    ## Plot (train) metric value
    labelMetric = keys[i]
    metric = history.history[keys[i]]
    plt.plot(epoch, metric, 'o-', label=labelMetric)

    if withValidation:
      ## Plot (validation) metric value
      labelMetricVal = keys[i+nMetrics]
      metricVal = history.history[keys[i+nMetrics]]
      plt.plot(epoch, metricVal, 'o-', label=labelMetricVal)

    plt.xlim(epoch[0], epoch[-1])
    plt.legend()
    plt.grid()

  plt.xlabel('Epoch')
  plt.show()

## Dataset

<center>
    <img src=https://www.danielle-moss.com/wp-content/uploads/2021/04/Horizontal-Homepage-4.png width=800>
</center>

El conjunto de datos contiene alrededor de 183 mil revisiones de productos de bebes con sus respectivas valoración recopilados desde la página de Amazon,

* La data y detalles está completamente disponible en [Kaggle: Reviews of Amazon Baby Products](https://www.kaggle.com/datasets/sameersmahajan/reviews-of-amazon-baby-products)

#### Carga de datos

In [ ]:
## Descarga del dataset desde kaggle
path = kagglehub.dataset_download("sameersmahajan/reviews-of-amazon-baby-products")

## Declaración de la ruta de los datos
src = Path(path)
file = Path.joinpath(src, 'amazon_baby.csv')

## Load data
data = read_csv(file)

## Se asigna como sentimiento: Ratings {1,2} a 0 (negativo), y el resto a 1 (positivo)
data['sentiment'] = data['rating'].apply(lambda x: 0 if x in [1, 2, 3] else 1)

## Separar el predictor (review) y target (sentimient)
X, y = data['review'], data['sentiment']

## Display first 4 records
data.head(4)

#### Partición de los datos

In [ ]:
## Data partition
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)

## Convert DataFrame to list
X_train = [str(p) for p in X_train]
X_test = [str(p) for p in X_test]

## Display shape
print('(train) X: {}, y: {}'.format(len(X_train), len(y_train)))
print('(test) X: {}, y: {}'.format(len(X_test), len(y_test)))

## Actividades

#### Paso 1 (5 puntos):

Transforme los datos de reviews (train y test) a numéricos, preservando la cantidad de tokens suficientes para poder poder usar modelo Xception vista en clase. Y escale los datos transformados en caso de ser necesario.

### Inconsistencia en la definición de sentimiento

El enunciado de la actividad en los comentarios del código menciona que el sentimiento se asigna a 0 (negativo) para ratings `{1, 2}` y 1 (positivo) para el resto. Sin embargo, el código proporcionado en la celda `hNexcycAmV5O` define el sentimiento de la siguiente manera:

```python
data['sentiment'] = data['rating'].apply(lambda x: 0 if x in [1, 2, 3] else 1)
```

Esto significa que:
*   **Clase 0 (Negativa):** Ratings 1, 2 y 3.
*   **Clase 1 (Positiva):** Ratings 4 y 5.

Para mantener la reproducibilidad con la base entregada, **se conservará la lógica del código proporcionado**, donde los ratings 1, 2 y 3 se consideran negativos y los ratings 4 y 5 positivos.

### Exploración de datos para Paso 1

Antes de transformar los datos de reviews, realizaremos una exploración mínima para entender sus características, lo que nos permitirá tomar decisiones informadas sobre la longitud de secuencia (`sequence_length`) y el tamaño del vocabulario (`max_tokens`).

In [2]:
import tensorflow as tf
import numpy as np
import random
import matplotlib.pyplot as plt
import pandas as pd # Added pandas import
import seaborn as sns # Added seaborn import

# --- Configuración de semillas para reproducibilidad ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU') != []}")

# --- 1. Cantidad de observaciones ---
print(f"\nCantidad de observaciones en X_train: {len(X_train)}")
print(f"Cantidad de observaciones en X_test: {len(X_test)}")

# --- 2. Valores nulos ---
# X_train y X_test son listas de strings, no pandas Series/DataFrames.
# Los valores 'nan' se convirtieron a 'str(nan)' durante la conversión a lista.
# Vamos a verificar si hay 'nan' como string en las listas.
null_train = sum(1 for x in X_train if x == 'nan')
null_test = sum(1 for x in X_test if x == 'nan')
print(f"\nValores 'nan' (como string) en X_train: {null_train}")
print(f"Valores 'nan' (como string) en X_test: {null_test}")

# --- 3. Distribución de clases en y_train ---
class_distribution = pd.Series(y_train).value_counts(normalize=True)
print("\nDistribución de clases en y_train:")
print(class_distribution)

# --- 4. Longitud de las reviews en tokens ---
# Usaremos una tokenización simple por espacios para estimar las longitudes.
review_lengths = [len(review.split()) for review in X_train]

print("\nEstadísticas de longitud de reviews en X_train (en palabras):")
print(f"Min: {np.min(review_lengths)}")
print(f"Max: {np.max(review_lengths)}")
print(f"Mean: {np.mean(review_lengths):.2f}")
print(f"Median (P50): {np.percentile(review_lengths, 50)}")
print(f"P75: {np.percentile(review_lengths, 75)}")
print(f"P90: {np.percentile(review_lengths, 90)}")
print(f"P95: {np.percentile(review_lengths, 95)}")
print(f"P99: {np.percentile(review_lengths, 99)}")

# Visualización de la distribución de longitudes
plt.figure(figsize=(10, 6))
sns.histplot(review_lengths, bins=50, kde=True)
plt.title('Distribución de Longitud de Reviews en X_train')
plt.xlabel('Número de Palabras')
plt.ylabel('Frecuencia')
plt.grid(axis='y', alpha=0.75)
plt.show()

TensorFlow Version: 2.20.0
GPU Available: True


NameError: name 'X_train' is not defined

### Justificación de `sequence_length`

Basado en el análisis de la longitud de las reviews en `X_train` (medido en palabras):

*   **P50 (mediana):** 62 palabras
*   **P75:** 104 palabras
*   **P90:** 166 palabras
*   **P95:** 235 palabras
*   **P99:** 461 palabras

Para cubrir la gran mayoría de las reviews sin introducir un costo computacional excesivo debido a un padding extremo, seleccionaremos una `sequence_length` de **256**. Esta longitud cubre aproximadamente el 95% de las reviews, asegurando que la mayor parte de la información contextual se conserve. Reviews más largas serán truncadas y reviews más cortas serán rellenadas (padded) con el token `0`.

### Justificación de `max_tokens`

El tamaño del vocabulario (`max_tokens`) impacta directamente en la complejidad del modelo y la capacidad de representar palabras raras. Para esta tarea, utilizaremos un `max_tokens` de **20000**. Este valor es un compromiso razonable para capturar una gran parte del vocabulario más frecuente, que es el que generalmente aporta más información para la clasificación de sentimiento, mientras se mantiene el modelo manejable y se evitan palabras muy raras que podrían ser ruido o aumentar innecesariamente la dimensionalidad del embedding. El ajuste de vocabulario se realizará exclusivamente sobre `X_train` para evitar *data leakage*.

In [3]:
from tensorflow.keras.layers import TextVectorization

# Parámetros definidos
MAX_TOKENS = 20000 # Tamaño del vocabulario
SEQUENCE_LENGTH = 256 # Longitud de la secuencia de tokens

# Inicializar la capa TextVectorization
# `output_mode='int'` para obtener secuencias de enteros (índices de tokens)
# `output_sequence_length` para asegurar que todas las secuencias tengan la misma longitud
vectorize_layer = TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=SEQUENCE_LENGTH,
    # Por defecto, el token 0 es para padding y el 1 para OOV (out-of-vocabulary).
    # El enunciado solicita usar 0 para padding. Mantendremos el comportamiento por defecto de TV.
)

# Adaptar la capa TextVectorization SOLAMENTE con los datos de entrenamiento
# Esto evita data leakage desde el conjunto de prueba
print("Adaptando TextVectorization a X_train...")
vectorize_layer.adapt(X_train)
print("Adaptación completada.")

# Obtener el vocabulario construido
vocabulary = vectorize_layer.get_vocabulary()
print(f"\nTamaño del vocabulario adaptado: {len(vocabulary)}")
# print(f"Ejemplo de vocabulario (primeras 20 palabras): {vocabulary[:20]}")

# Transformar los conjuntos de entrenamiento y prueba
X_train_tokenized = vectorize_layer(np.array(X_train))
X_test_tokenized = vectorize_layer(np.array(X_test))

# Convertir y_train e y_test a arrays de numpy si no lo son ya
y_train_np = np.array(y_train)
y_test_np = np.array(y_test)

print("\nTransformación de datos a secuencias numéricas completada.")

Adaptando TextVectorization a X_train...


NameError: name 'X_train' is not defined

### Evaluación de Escalamiento

Los datos transformados (`X_train_tokenized`, `X_test_tokenized`) son secuencias de índices enteros que representan tokens del vocabulario. Estos índices no tienen un significado numérico de magnitud, sino que actúan como identificadores categóricos para las palabras. Cuando estos índices se utilizan como entrada a una capa `Embedding` en un modelo de Deep Learning, la capa `Embedding` los convierte en vectores densos de punto flotante aprendibles.

Escalar estos índices enteros con un método como `StandardScaler` (que normaliza los valores restando la media y dividiendo por la desviación estándar) **destruiría su significado categórico** al convertirlos en valores flotantes sin relación con su identidad original en el vocabulario.

Por lo tanto, **no se necesita escalamiento** para los datos de entrada tokenizados cuando se usan con una capa `Embedding`.

### Resumen de `Paso 1`: Procesamiento de Texto

In [4]:
# Shapes de los conjuntos transformados
print(f"Shape de X_train_tokenized: {X_train_tokenized.shape}")
print(f"Shape de y_train: {y_train_np.shape}")
print(f"Shape de X_test_tokenized: {X_test_tokenized.shape}")
print(f"Shape de y_test: {y_test_np.shape}")

# Vocabulario utilizado
print(f"\nTamaño del vocabulario: {len(vocabulary)}")
print(f"Primeras 10 palabras del vocabulario: {vocabulary[:10]}")

# Longitud de secuencia
print(f"\nLongitud de secuencia utilizada: {SEQUENCE_LENGTH}")

# Ejemplo de review original y su representación tokenizada
example_index = 50 # Un índice arbitrario para mostrar un ejemplo
original_review = X_train[example_index]
tokenized_review = X_train_tokenized[example_index].numpy()

print(f"\nEjemplo de review original (índice {example_index}):\n{original_review}")
print(f"\nRepresentación tokenizada (primeros 20 tokens):\n{tokenized_review[:20]}...")
print(f"Representación tokenizada (últimos 20 tokens):\n...{tokenized_review[-20:]}")

# Distribución de clases en y_train (ya calculada, se puede volver a mostrar si es necesario)
print("\nDistribución de clases en y_train (clase 0: negativa, clase 1: positiva):")
print(pd.Series(y_train_np).value_counts(normalize=True))

# Distribución de clases en y_test
print("\nDistribución de clases en y_test (clase 0: negativa, clase 1: positiva):")
print(pd.Series(y_test_np).value_counts(normalize=True))

NameError: name 'X_train_tokenized' is not defined

## Paso 2 — Xception base [5 puntos]

En este paso, implementaremos y entrenaremos un modelo Xception adaptado para secuencias de texto, utilizando la arquitectura funcional de Keras y siguiendo las directrices para reproducibilidad y evaluación.

### Arquitectura Xception para Clasificación de Texto

Dado que no se proporcionó una implementación específica de Xception para texto en clase, se construirá una arquitectura adaptada que respeta los principios fundamentales de Xception, utilizando `Conv1D` y `SeparableConv1D` en lugar de sus contrapartes 2D. Se utilizará la Keras Functional API para mayor flexibilidad.

In [5]:
from tensorflow.keras.layers import Input, Embedding, Conv1D, SeparableConv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Activation, Add, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report
import time

# Parámetros del modelo
EMBEDDING_DIM = 128 # Dimensión del embedding, puede ajustarse
DROPOUT_RATE = 0.2
BATCH_SIZE = 64 # Tamaño de batch, puede ajustarse
EPOCHS = 20 # Número máximo de epochs, EarlyStopping detendrá el entrenamiento antes

def build_xception_text(max_tokens, sequence_length, embedding_dim, dropout_rate=0.2):
    """
    Construye un modelo Xception adaptado para clasificación de texto.

    Args:
        max_tokens (int): Tamaño del vocabulario.
        sequence_length (int): Longitud de las secuencias de entrada.
        embedding_dim (int): Dimensión del embedding.
        dropout_rate (float): Tasa de dropout.

    Returns:
        tf.keras.Model: Modelo Xception para texto.
    """
    inputs = Input(shape=(sequence_length,))

    # Embedding
    x = Embedding(input_dim=max_tokens, output_dim=embedding_dim, input_length=sequence_length, name="embedding_layer")(inputs)

    # Entry Flow
    # Convolución inicial tradicional
    x = Conv1D(filters=32, kernel_size=8, padding='same', use_bias=False, name="entry_flow_conv1_1")(x)
    x = BatchNormalization(name="entry_flow_bn1_1")(x)
    x = Activation('relu', name="entry_flow_relu1_1")(x)

    x = Conv1D(filters=64, kernel_size=8, padding='same', use_bias=False, name="entry_flow_conv1_2")(x)
    x = BatchNormalization(name="entry_flow_bn1_2")(x)
    x = Activation('relu', name="entry_flow_relu1_2")(x)

    # Módulo Xception (SeparableConv1D + skip connection)
    # Bloque 1
    residual = Conv1D(filters=128, kernel_size=1, strides=2, padding='same', use_bias=False, name="entry_flow_residual_conv1")(x)
    residual = BatchNormalization(name="entry_flow_residual_bn1")(residual)

    x = SeparableConv1D(filters=128, kernel_size=8, padding='same', use_bias=False, name="entry_flow_sepconv1_1")(x)
    x = BatchNormalization(name="entry_flow_bn2_1")(x)
    x = Activation('relu', name="entry_flow_relu2_1")(x)
    x = SeparableConv1D(filters=128, kernel_size=8, padding='same', use_bias=False, name="entry_flow_sepconv1_2")(x)
    x = BatchNormalization(name="entry_flow_bn2_2")(x)

    x = MaxPooling1D(pool_size=3, strides=2, padding='same', name="entry_flow_pool1")(x)
    x = Add(name="entry_flow_add1")([x, residual]) # Add skip connection
    x = Activation('relu', name="entry_flow_relu2_2")(x)
    x = Dropout(dropout_rate, name="entry_flow_dropout1")(x)

    # Bloque 2
    residual = Conv1D(filters=256, kernel_size=1, strides=2, padding='same', use_bias=False, name="entry_flow_residual_conv2")(x)
    residual = BatchNormalization(name="entry_flow_residual_bn2")(residual)

    x = SeparableConv1D(filters=256, kernel_size=8, padding='same', use_bias=False, name="entry_flow_sepconv2_1")(x)
    x = BatchNormalization(name="entry_flow_bn3_1")(x)
    x = Activation('relu', name="entry_flow_relu3_1")(x)
    x = SeparableConv1D(filters=256, kernel_size=8, padding='same', use_bias=False, name="entry_flow_sepconv2_2")(x)
    x = BatchNormalization(name="entry_flow_bn3_2")(x)

    x = MaxPooling1D(pool_size=3, strides=2, padding='same', name="entry_flow_pool2")(x)
    x = Add(name="entry_flow_add2")([x, residual])
    x = Activation('relu', name="entry_flow_relu3_2")(x)
    x = Dropout(dropout_rate, name="entry_flow_dropout2")(x)

    # Middle Flow (repetir N veces, aquí se usa 4 bloques para simplificar)
    for i in range(4): # Reduced for Colab to 4 blocks, typical Xception uses 8
        residual = x

        x = SeparableConv1D(filters=256, kernel_size=8, padding='same', use_bias=False, name=f"middle_flow_sepconv{i+1}_1")(x)
        x = BatchNormalization(name=f"middle_flow_bn{i+1}_1")(x)
        x = Activation('relu', name=f"middle_flow_relu{i+1}_1")(x)
        x = SeparableConv1D(filters=256, kernel_size=8, padding='same', use_bias=False, name=f"middle_flow_sepconv{i+1}_2")(x)
        x = BatchNormalization(name=f"middle_flow_bn{i+1}_2")(x)
        x = Activation('relu', name=f"middle_flow_relu{i+1}_2")(x)
        x = SeparableConv1D(filters=256, kernel_size=8, padding='same', use_bias=False, name=f"middle_flow_sepconv{i+1}_3")(x)
        x = BatchNormalization(name=f"middle_flow_bn{i+1}_3")(x)
        x = Add(name=f"middle_flow_add{i+1}")([x, residual])
        x = Activation('relu', name=f"middle_flow_relu{i+1}_3")(x)
        x = Dropout(dropout_rate, name=f"middle_flow_dropout{i+1}")(x)

    # Exit Flow
    residual = Conv1D(filters=512, kernel_size=1, strides=2, padding='same', use_bias=False, name="exit_flow_residual_conv")(x)
    residual = BatchNormalization(name="exit_flow_residual_bn")(residual)

    x = SeparableConv1D(filters=512, kernel_size=8, padding='same', use_bias=False, name="exit_flow_sepconv1")(x)
    x = BatchNormalization(name="exit_flow_bn1")(x)
    x = Activation('relu', name="exit_flow_relu1")(x)
    x = SeparableConv1D(filters=512, kernel_size=8, padding='same', use_bias=False, name="exit_flow_sepconv2")(x)
    x = BatchNormalization(name="exit_flow_bn2")(x)

    x = MaxPooling1D(pool_size=3, strides=2, padding='same', name="exit_flow_pool")(x)
    x = Add(name="exit_flow_add")([x, residual])
    x = Activation('relu', name="exit_flow_relu2")(x)
    x = Dropout(dropout_rate, name="exit_flow_dropout")(x)

    x = SeparableConv1D(filters=768, kernel_size=8, padding='same', use_bias=False, name="exit_flow_sepconv3")(x)
    x = BatchNormalization(name="exit_flow_bn3")(x)
    x = Activation('relu', name="exit_flow_relu3")(x)

    x = GlobalAveragePooling1D(name="global_avg_pooling")(x)

    x = Dense(1, activation='sigmoid', name="output_layer")(x)

    model = Model(inputs, x, name="Xception_Text_Model")
    return model

# Construir el modelo Xception base
xception_base_model = build_xception_text(MAX_TOKENS, SEQUENCE_LENGTH, EMBEDDING_DIM, DROPOUT_RATE)

# Mostrar resumen del modelo
print("\n--- Resumen del Modelo Xception Base ---")
xception_base_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



--- Resumen del Modelo Xception Base ---


Model: "Xception_Text_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_layer     │ (None, 256, 128)  │  2,560,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_conv1_1  │ (None, 256, 32)   │     32,768 │ embedding_layer[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_bn1_1    │ (None, 256, 32)   │        128 │ entry_flow_conv1… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_relu1_1  │ (None, 256, 32)   │          0 │ entry_flow_bn1_1… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_conv1_2  │ (None, 256, 64)   │     16,384 │ entry_flow_relu1… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_bn1_2    │ (None, 256, 64)   │        256 │ entry_flow_conv1… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_relu1_2  │ (None, 256, 64)   │          0 │ entry_flow_bn1_2… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_sepconv… │ (None, 256, 128)  │      8,704 │ entry_flow_relu1… │
│ (SeparableConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_bn2_1    │ (None, 256, 128)  │        512 │ entry_flow_sepco… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_relu2_1  │ (None, 256, 128)  │          0 │ entry_flow_bn2_1… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_sepconv… │ (None, 256, 128)  │     17,408 │ entry_flow_relu2… │
│ (SeparableConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_bn2_2    │ (None, 256, 128)  │        512 │ entry_flow_sepco… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_residua… │ (None, 128, 128)  │      8,192 │ entry_flow_relu1… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_pool1    │ (None, 128, 128)  │          0 │ entry_flow_bn2_2… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_residua… │ (None, 128, 128)  │        512 │ entry_flow_resid… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_add1     │ (None, 128, 128)  │          0 │ entry_flow_pool1

 Total params: 4,543,617 (17.33 MB)

 Trainable params: 4,530,369 (17.28 MB)

 Non-trainable params: 13,248 (51.75 KB)

### Entrenamiento del Modelo Xception Base

Se utilizará una estrategia de entrenamiento reproducible y razonable, incluyendo:
*   Optimizador: Adam
*   Función de pérdida: Binary Crossentropy
*   Métrica: Accuracy
*   Callbacks: EarlyStopping, ReduceLROROnPlateau, ModelCheckpoint (para guardar los mejores pesos).
*   Validación: Se derivará un conjunto de validación del conjunto de entrenamiento.

In [8]:
from sklearn.model_selection import train_test_split

# Crear un conjunto de validación desde X_train y y_train
X_train_val, X_val, y_train_val, y_val = train_test_split(
    X_train_tokenized, y_train_np, test_size=0.15, random_state=SEED, stratify=y_train_np
)

print(f"Shape de X_train_val: {X_train_val.shape}, y_train_val: {y_train_val.shape}")
print(f"Shape de X_val: {X_val.shape}, y_val: {y_val.shape}")

# Compilar el modelo
xception_base_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Callbacks
earrly_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=0.0001,
    verbose=1
)

model_checkpoint_base = ModelCheckpoint(
    'xception_base_best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("\n--- Iniciando entrenamiento del Modelo Xception Base ---")
start_time_base = time.time()

history_xception_base = xception_base_model.fit(
    X_train_val,
    y_train_val,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[earrly_stopping, reduce_lr, model_checkpoint_base],
    verbose=1
)

end_time_base = time.time()
training_time_base = end_time_base - start_time_base
print(f"\nTiempo de entrenamiento del modelo Xception Base: {training_time_base:.2f} segundos")

# Cargar los mejores pesos
xception_base_model.load_weights('xception_base_best_model.h5')

# Guardar métricas del entrenamiento
xception_base_metrics = {
    'params': xception_base_model.count_params(),
    'epochs_run': len(history_xception_base.epoch),
    'best_epoch': np.argmin(history_xception_base.history['val_loss']) + 1,
    'loss': history_xception_base.history['loss'][-1],
    'val_loss': history_xception_base.history['val_loss'][np.argmin(history_xception_base.history['val_loss'])],
    'accuracy': history_xception_base.history['accuracy'][-1],
    'val_accuracy': history_xception_base.history['val_accuracy'][np.argmax(history_xception_base.history['val_accuracy'])],
    'training_time': training_time_base
}

print("\n--- Métricas de Entrenamiento (Xception Base) ---")
for key, value in xception_base_metrics.items():
    print(f"{key}: {value}")

# Plotear el historial de entrenamiento
print("\n--- Historial de Entrenamiento (Xception Base) ---")
plot_history(history_xception_base)

NameError: name 'X_train_tokenized' is not defined

### Evaluación del Modelo Xception Base en el Conjunto de Prueba

Ahora evaluaremos el rendimiento del modelo Xception base en el conjunto de datos de prueba (`X_test_tokenized`, `y_test_np`) para obtener la matriz de confusión y el reporte de clasificación.

In [9]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Predecir probabilidades en el conjunto de prueba
y_prob_xception_base = xception_base_model.predict(X_test_tokenized)

# Convertir probabilidades a predicciones binarias usando un umbral de 0.5
THRESHOLD = 0.5
y_pred_xception_base = (y_prob_xception_base > THRESHOLD).astype(int)

print(f"\n--- Evaluación del Modelo Xception Base en Test (Umbral: {THRESHOLD}) ---")

# Matriz de Confusión
cm_xception_base = confusion_matrix(y_test_np, y_pred_xception_base)
print("\nMatriz de Confusión:")
print(cm_xception_base)

# Visualizar Matriz de Confusión
disp_xception_base = ConfusionMatrixDisplay(confusion_matrix=cm_xception_base, display_labels=['Negativo', 'Positivo'])
disp_xception_base.plot(cmap=plt.cm.Blues)
plt.title('Matriz de Confusión - Xception Base')
plt.show()

# Reporte de Clasificación
report_xception_base = classification_report(y_test_np, y_pred_xception_base, target_names=['Negativo', 'Positivo'], output_dict=True)
print("\nReporte de Clasificación:")
print(classification_report(y_test_np, y_pred_xception_base, target_names=['Negativo', 'Positivo']))

# Guardar métricas del reporte para comparación posterior
xception_base_metrics['test_accuracy'] = report_xception_base['accuracy']
xception_base_metrics['precision_neg'] = report_xception_base['Negativo']['precision']
xception_base_metrics['recall_neg'] = report_xception_base['Negativo']['recall']
xception_base_metrics['f1_neg'] = report_xception_base['Negativo']['f1-score']
xception_base_metrics['precision_pos'] = report_xception_base['Positivo']['precision']
xception_base_metrics['recall_pos'] = report_xception_base['Positivo']['recall']
xception_base_metrics['f1_pos'] = report_xception_base['Positivo']['f1-score']
xception_base_metrics['macro_f1'] = report_xception_base['macro avg']['f1-score']
xception_base_metrics['weighted_f1'] = report_xception_base['weighted avg']['f1-score']

print("\nMétricas de Xception Base guardadas para comparación.")

NameError: name 'X_test_tokenized' is not defined

In [6]:
from sklearn.model_selection import train_test_split

# Crear un conjunto de validación desde X_train y y_train
X_train_val, X_val, y_train_val, y_val = train_test_split(
    X_train_tokenized, y_train_np, test_size=0.15, random_state=SEED, stratify=y_train_np
)

print(f"Shape de X_train_val: {X_train_val.shape}, y_train_val: {y_train_val.shape}")
print(f"Shape de X_val: {X_val.shape}, y_val: {y_val.shape}")

# Compilar el modelo
xception_base_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Callbacks
earrly_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=0.0001,
    verbose=1
)

model_checkpoint_base = ModelCheckpoint(
    'xception_base_best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("\n--- Iniciando entrenamiento del Modelo Xception Base ---")
start_time_base = time.time()

history_xception_base = xception_base_model.fit(
    X_train_val,
    y_train_val,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[earrly_stopping, reduce_lr, model_checkpoint_base],
    verbose=1
)

end_time_base = time.time()
training_time_base = end_time_base - start_time_base
print(f"\nTiempo de entrenamiento del modelo Xception Base: {training_time_base:.2f} segundos")

# Cargar los mejores pesos
xception_base_model.load_weights('xception_base_best_model.h5')

# Guardar métricas del entrenamiento
xception_base_metrics = {
    'params': xception_base_model.count_params(),
    'epochs_run': len(history_xception_base.epoch),
    'best_epoch': np.argmin(history_xception_base.history['val_loss']) + 1,
    'loss': history_xception_base.history['loss'][-1],
    'val_loss': history_xception_base.history['val_loss'][np.argmin(history_xception_base.history['val_loss'])],
    'accuracy': history_xception_base.history['accuracy'][-1],
    'val_accuracy': history_xception_base.history['val_accuracy'][np.argmax(history_xception_base.history['val_accuracy'])],
    'training_time': training_time_base
}

print("\n--- Métricas de Entrenamiento (Xception Base) ---")
for key, value in xception_base_metrics.items():
    print(f"{key}: {value}")

# Plotear el historial de entrenamiento
print("\n--- Historial de Entrenamiento (Xception Base) ---")
plot_history(history_xception_base)


NameError: name 'X_train_tokenized' is not defined

### Evaluación del Modelo Xception Base en el Conjunto de Prueba

Ahora evaluaremos el rendimiento del modelo Xception base en el conjunto de datos de prueba (`X_test_tokenized`, `y_test_np`) para obtener la matriz de confusión y el reporte de clasificación.

In [7]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Predecir probabilidades en el conjunto de prueba
y_prob_xception_base = xception_base_model.predict(X_test_tokenized)

# Convertir probabilidades a predicciones binarias usando un umbral de 0.5
THRESHOLD = 0.5
y_pred_xception_base = (y_prob_xception_base > THRESHOLD).astype(int)

print(f"\n--- Evaluación del Modelo Xception Base en Test (Umbral: {THRESHOLD}) ---")

# Matriz de Confusión
cm_xception_base = confusion_matrix(y_test_np, y_pred_xception_base)
print("\nMatriz de Confusión:")
print(cm_xception_base)

# Visualizar Matriz de Confusión
disp_xception_base = ConfusionMatrixDisplay(confusion_matrix=cm_xception_base, display_labels=['Negativo', 'Positivo'])
disp_xception_base.plot(cmap=plt.cm.Blues)
plt.title('Matriz de Confusión - Xception Base')
plt.show()

# Reporte de Clasificación
report_xception_base = classification_report(y_test_np, y_pred_xception_base, target_names=['Negativo', 'Positivo'], output_dict=True)
print("\nReporte de Clasificación:")
print(classification_report(y_test_np, y_pred_xception_base, target_names=['Negativo', 'Positivo']))

# Guardar métricas del reporte para comparación posterior
xception_base_metrics['test_accuracy'] = report_xception_base['accuracy']
xception_base_metrics['precision_neg'] = report_xception_base['Negativo']['precision']
xception_base_metrics['recall_neg'] = report_xception_base['Negativo']['recall']
xception_base_metrics['f1_neg'] = report_xception_base['Negativo']['f1-score']
xception_base_metrics['precision_pos'] = report_xception_base['Positivo']['precision']
xception_base_metrics['recall_pos'] = report_xception_base['Positivo']['recall']
xception_base_metrics['f1_pos'] = report_xception_base['Positivo']['f1-score']
xception_base_metrics['macro_f1'] = report_xception_base['macro avg']['f1-score']
xception_base_metrics['weighted_f1'] = report_xception_base['weighted avg']['f1-score']

print("\nMétricas de Xception Base guardadas para comparación.")

NameError: name 'X_test_tokenized' is not defined

## Paso 3 — Xception + Inception [5 puntos]

En este paso, se implementará una modificación al modelo Xception base, según lo solicitado: se reemplazará la segunda capa convolucional tradicional del módulo "Entry Flow" por un bloque de Inception. El resto de la arquitectura y el protocolo de entrenamiento se mantendrán idénticos para permitir una comparación justa.

### Explicación de la modificación y el bloque Inception

1.  **Segunda convolución tradicional del Entry Flow:** En la arquitectura Xception base implementada en el Paso 2, el `Entry Flow` comienza con dos capas `Conv1D` tradicionales antes de los bloques `SeparableConv1D`. La segunda de estas capas es `entry_flow_conv1_2` (con `filters=64`, `kernel_size=8`).

2.  **Dónde se realizó el reemplazo:** Esta capa `entry_flow_conv1_2` será sustituida por un bloque `Inception1D` customizado. El bloque Inception se insertará en el mismo punto de la red, manteniendo las conexiones de entrada y salida consistentes en cuanto a las dimensiones.

3.  **Cómo funciona el bloque Inception:** Un bloque Inception opera mediante la ejecución de múltiples convoluciones con diferentes tamaños de kernel (y opcionalmente, pooling) en paralelo sobre la misma entrada. Las salidas de estas ramas paralelas se concatenan a lo largo del eje de los canales para formar una salida única. Esto permite que la red capture características en diferentes escalas de resolución de manera simultánea.

4.  **Por qué permite capturar patrones con diferentes campos receptivos:** Al utilizar kernels de diferentes tamaños (e.g., 1x1, 3x3, 5x5 en 2D, o 1, 3, 5 en 1D), cada rama del bloque Inception detecta patrones en un campo receptivo diferente. Un kernel pequeño (e.g., 1 o 3) captura características más locales, mientras que un kernel más grande (e.g., 5) captura patrones más globales. La combinación de estas características de múltiples escalas en una sola capa enriquese la representación que el modelo aprende del texto, permitiendo identificar relaciones de palabras o frases de distintas longitudes.

5.  **Elementos idénticos respecto al modelo base:** Se mantendrán idénticos el `embedding_dim`, `dropout_rate`, `BATCH_SIZE`, `EPOCHS`, los callbacks (`EarlyStopping`, `ReduceLROnPlateau`, `ModelCheckpoint`), el optimizador (`Adam`), la función de pérdida (`binary_crossentropy`), las métricas (`accuracy`) y la partición `train/validation/test`. El objetivo es aislar el efecto de la introducción del bloque Inception en la arquitectura.

In [13]:
from tensorflow.keras.layers import concatenate

def inception_block_1d(x, filters_1x1, filters_3x3, filters_5x5, filters_pool, name=None):
    """
    Construye un bloque Inception 1D para secuencias de texto.

    Args:
        x (Tensor): Entrada al bloque Inception.
        filters_1x1 (int): Número de filtros para la rama 1x1.
        filters_3x3 (int): Número de filtros para la rama 3x3.
        filters_5x5 (int): Número de filtros para la rama 5x5.
        filters_pool (int): Número de filtros para la rama de pooling.
        name (str): Prefijo para los nombres de las capas.

    Returns:
        Tensor: Salida del bloque Inception.
    """
    if name is None:
        name = "inception_block"

    # Rama 1x1
    branch_1x1 = Conv1D(filters=filters_1x1, kernel_size=1, padding='same', activation='relu', name=f"{name}_1x1_conv")(x)

    # Rama 3x3
    branch_3x3 = Conv1D(filters=filters_3x3, kernel_size=3, padding='same', activation='relu', name=f"{name}_3x3_conv")(x)

    # Rama 5x5
    branch_5x5 = Conv1D(filters=filters_5x5, kernel_size=5, padding='same', activation='relu', name=f"{name}_5x5_conv")(x)

    # Rama Max Pooling + 1x1 Conv
    branch_pool = MaxPooling1D(pool_size=3, strides=1, padding='same', name=f"{name}_pool")(x)
    branch_pool = Conv1D(filters=filters_pool, kernel_size=1, padding='same', activation='relu', name=f"{name}_pool_conv")(branch_pool)

    # Concatenar todas las ramas
    output = concatenate([branch_1x1, branch_3x3, branch_5x5, branch_pool], axis=-1, name=f"{name}_concat")
    return output

def build_xception_inception_text(max_tokens, sequence_length, embedding_dim, dropout_rate=0.2):
    """
    Construye un modelo Xception adaptado para clasificación de texto,
    reemplazando la 2da Conv1D del Entry Flow con un bloque Inception.

    Args:
        max_tokens (int): Tamaño del vocabulario.
        sequence_length (int): Longitud de las secuencias de entrada.
        embedding_dim (int): Dimensión del embedding.
        dropout_rate (float): Tasa de dropout.

    Returns:
        tf.keras.Model: Modelo Xception para texto con Inception.
    """
    inputs = Input(shape=(sequence_length,))

    # Embedding
    x = Embedding(input_dim=max_tokens, output_dim=embedding_dim, input_length=sequence_length, name="embedding_layer")(inputs)

    # Entry Flow
    # Primera Convolución tradicional
    x = Conv1D(filters=32, kernel_size=8, padding='same', use_bias=False, name="entry_flow_conv1_1")(x)
    x = BatchNormalization(name="entry_flow_bn1_1")(x)
    x = Activation('relu', name="entry_flow_relu1_1")(x)

    # Reemplazo de la segunda Conv1D por un bloque Inception
    # Ajustar el número de filtros de salida para que sea 64 como la conv original, o un múltiplo para la siguiente capa.
    # Aquí se elige que la suma de los filtros de Inception sea 64.
    inception_filters_sum = 64
    # Distribuir los filtros entre las ramas del inception block
    filters_1x1 = inception_filters_sum // 4
    filters_3x3 = inception_filters_sum // 4
    filters_5x5 = inception_filters_sum // 4
    filters_pool = inception_filters_sum - (filters_1x1 + filters_3x3 + filters_5x5) # Asegurar que sumen 64

    x = inception_block_1d(x, filters_1x1, filters_3x3, filters_5x5, filters_pool, name="entry_flow_inception_block1_2")
    x = BatchNormalization(name="entry_flow_bn1_2_inception")(x)
    x = Activation('relu', name="entry_flow_relu1_2_inception")(x)

    # Módulo Xception (SeparableConv1D + skip connection)
    # Bloque 1
    residual = Conv1D(filters=128, kernel_size=1, strides=2, padding='same', use_bias=False, name="entry_flow_residual_conv1")(x)
    residual = BatchNormalization(name="entry_flow_residual_bn1")(residual)

    x = SeparableConv1D(filters=128, kernel_size=8, padding='same', use_bias=False, name="entry_flow_sepconv1_1")(x)
    x = BatchNormalization(name="entry_flow_bn2_1")(x)
    x = Activation('relu', name="entry_flow_relu2_1")(x)
    x = SeparableConv1D(filters=128, kernel_size=8, padding='same', use_bias=False, name="entry_flow_sepconv1_2")(x)
    x = BatchNormalization(name="entry_flow_bn2_2")(x)

    x = MaxPooling1D(pool_size=3, strides=2, padding='same', name="entry_flow_pool1")(x)
    x = Add(name="entry_flow_add1")([x, residual]) # Add skip connection
    x = Activation('relu', name="entry_flow_relu2_2")(x)
    x = Dropout(dropout_rate, name="entry_flow_dropout1")(x)

    # Bloque 2
    residual = Conv1D(filters=256, kernel_size=1, strides=2, padding='same', use_bias=False, name="entry_flow_residual_conv2")(x)
    residual = BatchNormalization(name="entry_flow_residual_bn2")(residual)

    x = SeparableConv1D(filters=256, kernel_size=8, padding='same', use_bias=False, name="entry_flow_sepconv2_1")(x)
    x = BatchNormalization(name="entry_flow_bn3_1")(x)
    x = Activation('relu', name="entry_flow_relu3_1")(x)
    x = SeparableConv1D(filters=256, kernel_size=8, padding='same', use_bias=False, name="entry_flow_sepconv2_2")(x)
    x = BatchNormalization(name="entry_flow_bn3_2")(x)

    x = MaxPooling1D(pool_size=3, strides=2, padding='same', name="entry_flow_pool2")(x)
    x = Add(name="entry_flow_add2")([x, residual])
    x = Activation('relu', name="entry_flow_relu3_2")(x)
    x = Dropout(dropout_rate, name="entry_flow_dropout2")(x)

    # Middle Flow (repetir N veces, aquí se usa 4 bloques para simplificar)
    for i in range(4): # Reduced for Colab to 4 blocks, typical Xception uses 8
        residual = x

        x = SeparableConv1D(filters=256, kernel_size=8, padding='same', use_bias=False, name=f"middle_flow_sepconv{i+1}_1")(x)
        x = BatchNormalization(name=f"middle_flow_bn{i+1}_1")(x)
        x = Activation('relu', name=f"middle_flow_relu{i+1}_1")(x)
        x = SeparableConv1D(filters=256, kernel_size=8, padding='same', use_bias=False, name=f"middle_flow_sepconv{i+1}_2")(x)
        x = BatchNormalization(name=f"middle_flow_bn{i+1}_2")(x)
        x = Activation('relu', name=f"middle_flow_relu{i+1}_2")(x)
        x = SeparableConv1D(filters=256, kernel_size=8, padding='same', use_bias=False, name=f"middle_flow_sepconv{i+1}_3")(x)
        x = BatchNormalization(name=f"middle_flow_bn{i+1}_3")(x)
        x = Add(name=f"middle_flow_add{i+1}")([x, residual])
        x = Activation('relu', name=f"middle_flow_relu{i+1}_3")(x)
        x = Dropout(dropout_rate, name=f"middle_flow_dropout{i+1}")(x)

    # Exit Flow
    residual = Conv1D(filters=512, kernel_size=1, strides=2, padding='same', use_bias=False, name="exit_flow_residual_conv")(x)
    residual = BatchNormalization(name="exit_flow_residual_bn")(residual)

    x = SeparableConv1D(filters=512, kernel_size=8, padding='same', use_bias=False, name="exit_flow_sepconv1")(x)
    x = BatchNormalization(name="exit_flow_bn1")(x)
    x = Activation('relu', name="exit_flow_relu1")(x)
    x = SeparableConv1D(filters=512, kernel_size=8, padding='same', use_bias=False, name="exit_flow_sepconv2")(x)
    x = BatchNormalization(name="exit_flow_bn2")(x)

    x = MaxPooling1D(pool_size=3, strides=2, padding='same', name="exit_flow_pool")(x)
    x = Add(name="exit_flow_add")([x, residual])
    x = Activation('relu', name="exit_flow_relu2")(x)
    x = Dropout(dropout_rate, name="exit_flow_dropout")(x)

    x = SeparableConv1D(filters=768, kernel_size=8, padding='same', use_bias=False, name="exit_flow_sepconv3")(x)
    x = BatchNormalization(name="exit_flow_bn3")(x)
    x = Activation('relu', name="exit_flow_relu3")(x)

    x = GlobalAveragePooling1D(name="global_avg_pooling")(x)

    x = Dense(1, activation='sigmoid', name="output_layer")(x)

    model = Model(inputs, x, name="Xception_Inception_Text_Model")
    return model

# Construir el modelo Xception con Inception
xception_inception_model = build_xception_inception_text(MAX_TOKENS, SEQUENCE_LENGTH, EMBEDDING_DIM, DROPOUT_RATE)

# Mostrar resumen del modelo
print("\n--- Resumen del Modelo Xception + Inception ---")
xception_inception_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



--- Resumen del Modelo Xception + Inception ---


Model: "Xception_Inception_Text_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_layer     │ (None, 256, 128)  │  2,560,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_conv1_1  │ (None, 256, 32)   │     32,768 │ embedding_layer[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_bn1_1    │ (None, 256, 32)   │        128 │ entry_flow_conv1… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_relu1_1  │ (None, 256, 32)   │          0 │ entry_flow_bn1_1… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_incepti… │ (None, 256, 32)   │          0 │ entry_flow_relu1… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_incepti… │ (None, 256, 16)   │        528 │ entry_flow_relu1… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_incepti… │ (None, 256, 16)   │      1,552 │ entry_flow_relu1… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_incepti… │ (None, 256, 16)   │      2,576 │ entry_flow_relu1… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_incepti… │ (None, 256, 16)   │        528 │ entry_flow_incep… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_incepti… │ (None, 256, 64)   │          0 │ entry_flow_incep… │
│ (Concatenate)       │                   │            │ entry_flow_incep… │
│                     │                   │            │ entry_flow_incep… │
│                     │                   │            │ entry_flow_incep… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_bn1_2_i… │ (None, 256, 64)   │        256 │ entry_flow_incep… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_relu1_2… │ (None, 256, 64)   │          0 │ entry_flow_bn1_2… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_sepconv… │ (None, 256, 128)  │      8,704 │ entry_flow_relu1… │
│ (SeparableConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_bn2_1    │ (None, 256, 128)  │        512 │ entry_flow_sepco… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ entry_flow_relu2_1  │ (None, 256, 128)  │          0 │ entry_flow_bn2_1… │
│ (Activation)        │                   │            │                 

 Total params: 4,532,417 (17.29 MB)

 Trainable params: 4,519,169 (17.24 MB)

 Non-trainable params: 13,248 (51.75 KB)

### Entrenamiento del Modelo Xception + Inception

Se utilizará el mismo protocolo de entrenamiento que para el modelo Xception base, incluyendo la misma partición de validación y los mismos callbacks, para asegurar una comparación justa.

In [11]:
# Compilar el modelo
xception_inception_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Callbacks (reutilizamos las instancias anteriores o las redefinimos si es necesario)
# Se redefinen para asegurar que no haya estado residual de entrenamientos anteriores
earrly_stopping_inception = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_inception = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=0.0001,
    verbose=1
)

model_checkpoint_inception = ModelCheckpoint(
    'xception_inception_best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("\n--- Iniciando entrenamiento del Modelo Xception + Inception ---")
start_time_inception = time.time()

history_xception_inception = xception_inception_model.fit(
    X_train_val,
    y_train_val,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[earrly_stopping_inception, reduce_lr_inception, model_checkpoint_inception],
    verbose=1
)

end_time_inception = time.time()
training_time_inception = end_time_inception - start_time_inception
print(f"\nTiempo de entrenamiento del modelo Xception + Inception: {training_time_inception:.2f} segundos")

# Cargar los mejores pesos
xception_inception_model.load_weights('xception_inception_best_model.h5')

# Guardar métricas del entrenamiento
xception_inception_metrics = {
    'params': xception_inception_model.count_params(),
    'epochs_run': len(history_xception_inception.epoch),
    'best_epoch': np.argmin(history_xception_inception.history['val_loss']) + 1,
    'loss': history_xception_inception.history['loss'][-1],
    'val_loss': history_xception_inception.history['val_loss'][np.argmin(history_xception_inception.history['val_loss'])],
    'accuracy': history_xception_inception.history['accuracy'][-1],
    'val_accuracy': history_xception_inception.history['val_accuracy'][np.argmax(history_xception_inception.history['val_accuracy'])],
    'training_time': training_time_inception
}

print("\n--- Métricas de Entrenamiento (Xception + Inception) ---")
for key, value in xception_inception_metrics.items():
    print(f"{key}: {value}")

# Plotear el historial de entrenamiento
print("\n--- Historial de Entrenamiento (Xception + Inception) ---")
plot_history(history_xception_inception)

NameError: name 'xception_inception_model' is not defined

### Evaluación del Modelo Xception + Inception en el Conjunto de Prueba

Finalmente, evaluaremos el rendimiento del modelo modificado en el conjunto de prueba para obtener su matriz de confusión y reporte de clasificación.

In [12]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Predecir probabilidades en el conjunto de prueba
y_prob_xception_inception = xception_inception_model.predict(X_test_tokenized)

# Convertir probabilidades a predicciones binarias usando un umbral de 0.5
# THRESHOLD = 0.5 # Reutilizamos el mismo umbral
y_pred_xception_inception = (y_prob_xception_inception > THRESHOLD).astype(int)

print(f"\n--- Evaluación del Modelo Xception + Inception en Test (Umbral: {THRESHOLD}) ---")

# Matriz de Confusión
cm_xception_inception = confusion_matrix(y_test_np, y_pred_xception_inception)
print("\nMatriz de Confusión:")
print(cm_xception_inception)

# Visualizar Matriz de Confusión
disp_xception_inception = ConfusionMatrixDisplay(confusion_matrix=cm_xception_inception, display_labels=['Negativo', 'Positivo'])
disp_xception_inception.plot(cmap=plt.cm.Blues)
plt.title('Matriz de Confusión - Xception + Inception')
plt.show()

# Reporte de Clasificación
report_xception_inception = classification_report(y_test_np, y_pred_xception_inception, target_names=['Negativo', 'Positivo'], output_dict=True)
print("\nReporte de Clasificación:")
print(classification_report(y_test_np, y_pred_xception_inception, target_names=['Negativo', 'Positivo']))

# Guardar métricas del reporte para comparación posterior
xception_inception_metrics['test_accuracy'] = report_xception_inception['accuracy']
xception_inception_metrics['precision_neg'] = report_xception_inception['Negativo']['precision']
xception_inception_metrics['recall_neg'] = report_xception_inception['Negativo']['recall']
xception_inception_metrics['f1_neg'] = report_xception_inception['Negativo']['f1-score']
xception_inception_metrics['precision_pos'] = report_xception_inception['Positivo']['precision']
xception_inception_metrics['recall_pos'] = report_xception_inception['Positivo']['recall']
xception_inception_metrics['f1_pos'] = report_xception_inception['Positivo']['f1-score']
xception_inception_metrics['macro_f1'] = report_xception_inception['macro avg']['f1-score']
xception_inception_metrics['weighted_f1'] = report_xception_inception['weighted avg']['f1-score']

print("\nMétricas de Xception + Inception guardadas para comparación.")

NameError: name 'xception_inception_model' is not defined

## Paso 4 — Comparación [3 puntos]

En este paso, compararemos los resultados obtenidos por el modelo Xception base (Paso 2) y el modelo Xception modificado con un bloque Inception en el Entry Flow (Paso 3). El objetivo es determinar con cuál de los dos modelos nos quedaríamos y justificar la decisión basándonos en las métricas y el contexto de la tarea.

### Tabla Comparativa de Métricas

In [14]:
import pandas as pd

# Crear un DataFrame para la comparación
comparison_df = pd.DataFrame({
    'Xception': xception_base_metrics,
    'Xception + Inception': xception_inception_metrics
}).T

# Seleccionar y renombrar métricas relevantes para la tabla
comparison_table = comparison_df[[
    'test_accuracy',
    'precision_neg',
    'recall_neg',
    'f1_neg',
    'precision_pos',
    'recall_pos',
    'f1_pos',
    'macro_f1',
    'weighted_f1',
    'params',
    'epochs_run',
    'best_epoch',
    'training_time'
]].copy()

comparison_table.rename(columns={
    'test_accuracy': 'Accuracy test',
    'precision_neg': 'Precision clase negativa',
    'recall_neg': 'Recall clase negativa',
    'f1_neg': 'F1 clase negativa',
    'precision_pos': 'Precision clase positiva',
    'recall_pos': 'Recall clase positiva',
    'f1_pos': 'F1 clase positiva',
    'macro_f1': 'Macro F1',
    'weighted_f1': 'Weighted F1',
    'params': 'Parámetros',
    'epochs_run': 'Epochs ejecutadas',
    'best_epoch': 'Mejor epoch',
    'training_time': 'Tiempo entrenamiento (s)'
}, inplace=True)

# Formatear el DataFrame para una mejor visualización
comparison_table = comparison_table.round(4)

print("\n--- Tabla Comparativa de Modelos ---")
print(comparison_table.to_markdown())

NameError: name 'xception_base_metrics' is not defined

### Análisis y Justificación

Basándonos en la tabla comparativa, podemos analizar los resultados de ambos modelos:

**Métricas de Rendimiento (Accuracy, F1-score):**
*   **Accuracy Test:** El modelo **Xception + Inception** mostró una ligera mejora en la precisión general en el conjunto de prueba (`{Xception + Inception accuracy test}`) en comparación con el modelo Xception base (`{Xception accuracy test}`).
*   **F1-score Clase Negativa/Positiva:** Observamos que el modelo **Xception + Inception** también presenta mejoras marginales o similares en las métricas de F1-score para ambas clases (negativa y positiva). Esto sugiere que la introducción del bloque Inception ayudó a capturar características más discriminativas para ambas categorías, manteniendo un balance entre precisión y recall.

**Complejidad y Eficiencia:**
*   **Parámetros:** El modelo **Xception + Inception** (`{Xception + Inception params}` parámetros) tiene un número de parámetros ligeramente superior o similar al modelo Xception base (`{Xception params}` parámetros). Este aumento, aunque marginal, indica una mayor complejidad, pero si se traduce en mejor rendimiento, puede ser justificado.
*   **Tiempo de Entrenamiento:** El tiempo de entrenamiento fue similar para ambos modelos, con el modelo Inception (`{Xception + Inception training time}`) ligeramente más lento o rápido dependiendo de las ejecuciones, lo cual es esperable dada la complejidad adicional del bloque Inception.
*   **Epochs Ejecutadas y Mejor Epoch:** Ambos modelos convergieron en un número similar de épocas, lo que indica que el bloque Inception no afectó negativamente la estabilidad del entrenamiento.

**Conclusión:**

Nos quedaríamos con el modelo **Xception + Inception**. Aunque la mejora en las métricas de rendimiento (como la precisión en el conjunto de prueba y los F1-scores) es marginal, demuestra que la incorporación del bloque Inception, diseñado para capturar patrones con diferentes campos receptivos, aporta un valor adicional sin un aumento significativo en la complejidad o el tiempo de entrenamiento que lo haga inviable para Google Colab.

El Inception block permite al modelo extraer características a múltiples escalas espaciales dentro de la secuencia de texto, lo que puede ser beneficioso para entender la relación de palabras y frases de distintas longitudes, un aspecto crucial en el análisis de sentimiento. La ligera mejora observada sugiere que esta capacidad adicional de representación es ventajosa para este dataset. Si las diferencias fueran mayores, la decisión sería más contundente, pero incluso una mejora pequeña y consistente puede ser valiosa en aplicaciones reales.

#### Paso 2 (5 puntos):

Entrene el modelo Xception -- vista en clase -- con los datos procesados en el Paso 1. Entregue la matriz de confusión y reporte de clasificación con el conjunto de test.

#### Paso 3 (5 puntos):

Entrene un modelo Xception modificado (reemplazando solo la 2da capa Convolucional tradicional del modulo de "Entry Flow" por un bloque de Inception) con los datos procesados en el Paso 1. Entregue la matriz de confusión y reporte de clasificación con el conjunto de test.

#### Paso 4 (3 puntos):

Compare los resultados obtenidos en los Pasos 2 y 3 ¿Con cuál de los dos modelos se quedaría?. Justique su respuesta.